# Track 04a — RAG & Knowledge

### RAG(검색 증강 생성)란?

RAG는 **LLM이 답하기 전에 외부 지식을 먼저 검색하고, 그 검색 결과를 근거로 답하게 하는 패턴**입니다. 모델의 기억에만 의존하면 사실을 지어내기 쉽지만(hallucination), 질문마다 관련 문서를 임베딩 검색으로 찾아 컨텍스트에 넣어 주면 *근거 있는* 답을 만들 수 있습니다. 이 노트북에서는 그 과정을 **임베딩 → 검색 → 인용 답변 → 실패 복구** 순서로 직접 만들어 봅니다.

### 이 노트북에서 보여줄 것

| Session | 무엇을 배우나요 / 왜 중요한가요 |
|---|---|
| **1. 임베딩 & 유사도** | HR FAQ 50건을 벡터로 바꾼 뒤 코사인 유사도로 *의미가 가까운지* 비교합니다. 의미 검색, 중복 탐지, 선택적 클러스터링까지 확인하며 검색의 기본 부품을 익힙니다. |
| **2. in-memory RAG** | Postgres 없이 매뉴얼 스니펫 8건으로 `VectorRetrievalStrategy`와 `retrieve` 도구를 만들고, 검색한 청크를 근거 컨텍스트로 넣어 5개 질문에 **인용(`sources`) 포함 JSON 답변**을 받습니다. 자율 `ToolAgent`가 검색을 생략할 수 있는 이유도 함께 짚습니다. |
| **3. 실패 모드 복구** | 빈 검색, 미설정, 중복 청크, 컨텍스트 초과, 프롬프트 주입 5가지를 케이스 ID로 고정해 재현합니다. 운영에서 RAG가 흔히 실패하는 지점과 라이브러리 복구 API를 확인합니다. |

> 임베딩 서버가 없어도 **Session 3의 실패 복구 5종과 개념 설명 셀은 그대로 실행**되도록 구성했습니다. 인프라 없이도 핵심 흐름을 익힐 수 있고, 임베딩 서버가 있으면 Session 1·2의 실제 검색까지 실행할 수 있습니다. 여기에 `EXAONE_API_KEY`까지 있으면 Session 2의 인용 QA도 실행됩니다.

### 이 노트북을 마치면

- 임베딩·코사인 유사도로 **의미 검색**과 **중복 탐지**를 구현할 수 있습니다.
- `VectorRetrievalStrategy` + `build_rag_tool_registry`(`retrieve` 도구) + 검색 근거 컨텍스트로 **인용 포함 RAG QA**를 구성할 수 있습니다. 자율 `ToolAgent`가 검색을 건너뛸 수 있는 이유도 이해할 수 있습니다.
- RAG의 대표 **실패 5종**을 알아보고 `exaone.context_management`와 검색 전략 API로 복구할 수 있습니다.

**구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순서로 이어집니다.
**필요:** 개념·실패 복구 셀은 **키와 서버 없이** 실행됩니다. 실제 검색은 임베딩 서버가 필요하고, 인용 QA는 추가로 `EXAONE_API_KEY`가 필요합니다.
**산출물:** `_out/cluster_report.json`, `_out/manual_qa.json`, `_out/failure_recovery.json`


### (선택) 임베딩 · DB 인프라 셋업

**개념·수식·실패 복구(Session 3) 셀은 인프라 없이** 동작합니다. **실제 검색(Session 1·2)**을 확인하려면 임베딩 서버가 필요하고, **심화 `04b`(pgvector)**까지 진행하려면 Postgres가 추가로 필요합니다. 전제 조건은 **Docker** 설치입니다.

**① 임베딩만 — 04a Session 1·2용 (가장 가벼운 구성, Postgres·MS MARCO 불필요)**

```bash
cd infrastructure/setup
# (1) TEI 임베딩 모델만 _temp/에 받기 — step1의 [1/2] 단계
#     (step1_downloads.sh 전체를 실행하면 MS MARCO까지 받아 시간이 더 걸립니다)
python3 -c "from huggingface_hub import snapshot_download; snapshot_download('intfloat/multilingual-e5-small', cache_dir='../../_temp')"
# (2) 임베딩 컨테이너만 실행 (db 제외)
docker compose up -d --build embedding
# (3) 상태 확인 — 200이면 OK
curl http://localhost:8000/health
```

**② 임베딩 + Postgres(pgvector) — 04b 심화까지**

```bash
cd infrastructure/setup
./step1_downloads.sh      # TEI 모델 + MS MARCO 데이터
./step2_docker.sh         # db + embedding 실행 (docker compose up -d --build db embedding)
./step3_build_rag.sh      # MS MARCO → pgvector 인덱스
./step4_build_graph.sh    # graph(엔티티/관계)
docker compose ps         # db와 embedding 상태 확인
```

설정 후 **첫 셀(Setup)을 다시 실행**하면 `EMBED_OK = True`가 됩니다. 심화 환경까지 설정했다면 `POSTGRES`/`PGVECTOR`도 활성화됩니다. `.env`의 `EMBEDDING_BASE_URL`(기본 `http://localhost:8000`)·`POSTGRES_*`·`PGVECTOR_*` 값을 확인하세요. 자세한 내용은 [`infrastructure/setup/README.md`](../../infrastructure/setup/README.md)를 참고하세요.


In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate live LLM steps on API key presence.
# (kr) API 키 유무에 따라 라이브 LLM 단계 실행 여부를 정한다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
import exaone.integrations.embedding
import exaone.integrations.llm_env

TRACK04 = ROOT / "recipes" / "track04_rag_and_knowledge"
DATA = TRACK04 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
client = exaone.integrations.build_llm_from_env()

# (en) Probe the embedding server once; gate every embedding step on EMBED_OK.
# (kr) 임베딩 서버를 한 번 점검하고, 모든 임베딩 단계의 실행 여부를 EMBED_OK로 정한다.
EMB_URL = exaone.integrations.embedding.embedding_base_url_from_env()
EMBED_OK = exaone.integrations.embedding.embedding_server_reachable(EMB_URL)

print("exaone", exaone.__version__)
print("EMBEDDING_BASE_URL:", EMB_URL, "| EMBED_OK =", EMBED_OK)
print("HAS_API =", HAS_API)

# (en) When the embedding server is down, point to the setup guide cell at the top.
# (kr) 임베딩 서버가 꺼져 있으면 맨 위 셋업 가이드 셀을 알려준다.
if not EMBED_OK:
    print(
        "\n[안내] 임베딩 서버 미연결 — 라이브 검색(Session 1·2)을 보려면 임베딩 서버를 띄우세요."
    )
    print(
        "  맨 위 '(선택) 임베딩 · DB 인프라 셋업' 셀에 ① 임베딩만 / ② +Postgres 명령이 있습니다 (전제: Docker)."
    )
    print(
        "  요약: cd infrastructure/setup && docker compose up -d --build embedding  (TEI 모델 캐시 _temp 필요)"
    )
    print("  ※ 미연결이어도 Session 3 실패 복구·개념 셀은 정상 동작합니다.")

**출력 해석:** `model:`과 주요 경로가 출력되면 이 노트북에서 사용할 클라이언트와 데이터 위치가 준비된 것입니다.


## Session 1. 임베딩 & 유사도

**테스트 시나리오** — `korean_hr_faq.jsonl`의 HR FAQ 50건(`faq_records`)으로 코사인 유사도, 의미 검색, 선택적 클러스터링을 확인합니다.

임베딩은 텍스트를 벡터로 바꿔 *의미가 가까운지*를 수치로 비교하게 해 줍니다.

> **임베딩 서버가 필요합니다.** 서버가 없으면(`EMBED_OK=False`) 검색·중복·클러스터링 셀은 `[SKIP]`으로 건너뜁니다. 개념·수식 셀은 그대로 진행되며, 첫 셀이 실행 방법을 `[안내]`로 출력합니다. 자세한 실행 방법은 맨 위 **"(선택) 임베딩 · DB 인프라 셋업"** 셀을 참고하세요. 04a는 Postgres와 MS MARCO가 필요하지 않습니다.


### Session 1-1. 코사인 유사도란?

**개념:** 코사인 유사도는 두 벡터가 *같은 방향*을 향하는 정도를 재는 척도입니다. 임베딩에서는 보통 0~1 범위의 값으로 해석합니다. 벡터의 길이가 아니라 **방향(의미)**을 비교하므로 문서 길이에 덜 휘둘리고, *의미가 가까운지*를 판단하는 의미 검색·중복 탐지·추천의 기본 연산으로 쓰입니다.

**하는 일:** 코사인 유사도(`cosine_similarity`)와 질의 벡터로 상위 문서를 고르는 `rank_by_query_vector`를 정의합니다. Session 1-3의 검색이 이 두 함수를 사용합니다.

**수식:** 벡터 $a,\ b$의 코사인 유사도는 $\dfrac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert}$입니다. 정규화된(길이 1) 벡터라면 분모가 1이 되므로 **내적만으로** 비교할 수 있습니다.


In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)


def rank_by_query_vector(query_vec, doc_vectors, doc_ids, top_k):
    scored = [
        (doc_ids[i], cosine_similarity(query_vec, doc_vectors[i]))
        for i in range(len(doc_ids))
    ]
    scored.sort(key=lambda item: item[1], reverse=True)
    return scored[:top_k]


print("cosine_similarity / rank_by_query_vector 준비 완료")

**출력 해석:** 이 셀은 이후 단계에서 사용할 유사도 계산 함수와 랭킹 함수를 정의합니다. 에러 없이 실행되면 Session 1-3에서 의미 검색을 수행할 준비가 된 것입니다.


### Session 1-2. 데모 데이터(HR FAQ fixture) 로드

**fixture 소개:** `korean_hr_faq.jsonl`은 이 랩을 위해 만든 **합성 예시 데이터(fixture)**입니다. 가상 회사의 HR 제도를 다룬 한국어 FAQ 50건으로 구성되어 있으며, `leave`·`payroll`·`benefits`·`it`·`security` 5개 카테고리를 포함합니다. 각 항목은 `id`/`question`/`answer`/`category` 필드를 가집니다. 실제 사내 문서 대신 안전하게 검색을 시연하기 위한 샘플입니다.

**하는 일:** fixture를 읽어 카테고리와 질문 수를 출력하고, 임베딩 서버가 있으면 embedder를 준비합니다.

**입력:** `data/korean_hr_faq.jsonl`

**정상:** `시나리오:` + `FAQ count:` + `categories:` + 샘플 2줄

**의미:** Session 1-3의 검색·중복 탐지와 1-4의 클러스터링이 모두 이 FAQ를 사용합니다.


In [ ]:
faq_path = DATA / "korean_hr_faq.jsonl"
records = [
    json.loads(line)
    for line in faq_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print("시나리오: 한국어 HR FAQ", len(records), "건 (korean_hr_faq.jsonl)")
for r in records[:2]:
    print(f"  {r.get('id', '?')}:", (r.get("question") or r.get("q") or "")[:50])
texts = [f"{r['question']} {r['answer']}" for r in records]
labels = [r["category"] for r in records]
ids = [r["id"] for r in records]
print("FAQ count:", len(records))
print("categories:", sorted(set(labels)))

embedder = exaone.integrations.embedding.build_embedder_from_env() if EMBED_OK else None

**출력 해석:** FAQ 50건과 5개 카테고리가 출력되면 검색 실험에 사용할 데이터가 정상적으로 로드된 것입니다. 함께 출력되는 샘플 2줄은 각 항목이 `id`/`question`/`answer`/`category` 구조로 읽혔는지 확인하기 위한 미리보기입니다.


### Session 1-3. 의미 검색 + 중복 탐지

**하는 일:** 질의를 임베딩해 상위 FAQ 5건을 코사인 점수로 찾고(의미 검색), 모든 FAQ 쌍의 유사도가 임계값(`DUPLICATE_THRESHOLD = 0.92`) 이상이면 *중복 후보*로 셉니다.

**의미:** 질의와 *표현은 달라도 의미가 가까운* 문서를 찾는 것이 의미 검색입니다. 문서 쌍의 유사도가 높으면 거의 같은 항목이 중복 등록되었을 수 있으므로, FAQ 정리와 데이터 위생 점검의 기본 도구가 됩니다.


In [ ]:
DUPLICATE_THRESHOLD = 0.92
doc_vectors = []
DIM = exaone.integrations.embedding.embedding_dim_from_env()
duplicates = []

if EMBED_OK:
    doc_vectors = embedder.embed_batch(texts)
    DIM = len(doc_vectors[0])
    query = "연차는 몇 일인가요?"
    query_vec = embedder.embed_one(query)
    print("query:", query)
    for rank, (doc_id, score) in enumerate(
        rank_by_query_vector(query_vec, doc_vectors, ids, top_k=5), 1
    ):
        rec = next(r for r in records if r["id"] == doc_id)
        print(f"  {rank}. [{score:.3f}] {rec['question'][:50]}")

    for i in range(len(doc_vectors)):
        for j in range(i + 1, len(doc_vectors)):
            sim = cosine_similarity(doc_vectors[i], doc_vectors[j])
            if sim >= DUPLICATE_THRESHOLD:
                duplicates.append((ids[i], ids[j], sim))
    print(f"\nduplicate pairs (sim >= {DUPLICATE_THRESHOLD}):", len(duplicates))
else:
    # (en) Offline/concept tier: no embedding server, so skip the live search.
    # (kr) 오프라인 개념 단계: 임베딩 서버가 없으니 라이브 검색은 건너뛴다.
    print(
        "[SKIP] 임베딩 서버 미연결(EMBED_OK=False) — 의미 검색·중복 탐지는 서버 연결 시 실행됩니다."
    )

**출력 해석:** 질의 "연차는 몇 일인가요?"의 상위 결과를 보면:

- **의미(임베딩) 검색**이 동작합니다. "몇 일"이라는 단어가 그대로 없어도 *연차* 관련 FAQ(계산·이월·취소)가 상위에 나타나고, 그 아래로 *반차/반반차*가 이어집니다. 키워드 검색만으로는 놓치기 쉬운 항목입니다.
- **점수(0.8대)는 "동일"이 아니라 "관련"**을 뜻합니다. 1.0이 아니며, 연차 질문이 반차 질문보다 일관되게 조금 더 높은 점수를 받습니다.
- **검색은 정답이 아니라 근거 후보**를 모으는 단계입니다. 최종 답은 이 청크들을 LLM이 종합해 만듭니다(Session 2).
- `duplicate pairs (sim >= 0.92): 1`은 유사도 0.92 이상인 FAQ 쌍이 **1건** 있다는 뜻입니다. 거의 같은 질문이 중복 등록된 *정리 후보*이므로 FAQ 위생 점검 대상입니다. 임계값을 낮추면 후보가 늘고, 높이면 더 확실한 중복만 남습니다.

> 임베딩 서버가 없으면(`EMBED_OK=False`) 이 셀은 `[SKIP]` 안내만 출력합니다. 오프라인으로 개념만 확인하는 단계에서는 정상 동작입니다.


### Session 1-4. (선택) 클러스터링 — Agglomerative (cosine)

**하는 일:** (선택) `scikit-learn`이 있으면 50개 FAQ 임베딩을 Agglomerative 클러스터링(코사인 거리)으로 5개 군집으로 묶고, 군집별 개수를 출력합니다.

**의미:** 사람이 붙인 라벨을 보지 않고 임베딩만으로 비슷한 FAQ를 묶어 봅니다. 이를 통해 카테고리 구조가 임베딩 공간에 자연스럽게 드러나는지 확인합니다. `scikit-learn`이나 임베딩 서버가 없으면 이 선택 단계는 건너뜁니다.


In [ ]:
cluster_labels = []
if EMBED_OK and doc_vectors:
    try:
        import numpy as np
        from sklearn.cluster import AgglomerativeClustering
    except ImportError:
        print("[SKIP] 클러스터링은 scikit-learn이 필요합니다 (선택): pip install scikit-learn")
    else:
        model = AgglomerativeClustering(n_clusters=5, metric="cosine", linkage="average")
        cluster_labels = model.fit_predict(np.array(doc_vectors)).tolist()
        counts = {}
        for label in cluster_labels:
            counts[label] = counts.get(label, 0) + 1
        print("cluster sizes:", dict(sorted(counts.items())))

        # (en) Optional graph: PCA-2D scatter (color=cluster) + cluster x category heatmap.
        # (kr) (선택) 그래프: PCA-2D 산점도(색=군집) + 군집×카테고리 히트맵.
        try:
            import matplotlib.pyplot as plt
            from sklearn.decomposition import PCA
        except ImportError:
            print("[SKIP] 그래프는 matplotlib가 필요합니다 (선택): pip install matplotlib")
        else:
            # (en) Project 384-d embeddings to 2-D, then tally clusters against the human categories.
            # (kr) 384차원 임베딩을 2차원으로 투영하고, 군집을 사람이 매긴 카테고리와 대조한다.
            coords = PCA(n_components=2).fit_transform(np.array(doc_vectors))
            cats = sorted(set(labels))
            cat_pos = {c: i for i, c in enumerate(cats)}
            n_clusters = len(set(cluster_labels))
            matrix = np.zeros((n_clusters, len(cats)), dtype=int)
            for cl, cat in zip(cluster_labels, labels):
                matrix[cl][cat_pos[cat]] += 1

            # (en) Plot labels stay English: matplotlib's default font cannot render Hangul.
            # (kr) matplotlib 기본 폰트는 한글을 렌더링하지 못하므로 그래프 라벨은 영어로 둔다.
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
            ax1.scatter(coords[:, 0], coords[:, 1], c=cluster_labels, cmap="tab10", s=40)
            ax1.set_title("FAQ embeddings (PCA 2-D), color = cluster")
            ax1.set_xlabel("PC1")
            ax1.set_ylabel("PC2")
            im = ax2.imshow(matrix, cmap="Blues", aspect="auto")
            ax2.set_xticks(range(len(cats)))
            ax2.set_xticklabels(cats, rotation=45, ha="right")
            ax2.set_yticks(range(n_clusters))
            ax2.set_yticklabels([f"cluster {i}" for i in range(n_clusters)])
            ax2.set_title("cluster x category (cell = #FAQ)")
            for i in range(n_clusters):
                for j in range(len(cats)):
                    if matrix[i][j]:
                        ax2.text(j, i, matrix[i][j], ha="center", va="center")
            fig.colorbar(im, ax=ax2, fraction=0.046)
            fig.tight_layout()
            plt.show()
else:
    # (en) No embeddings available offline, so there is nothing to cluster.
    # (kr) 오프라인에서는 임베딩이 없어 군집화할 대상이 없다.
    print("[SKIP] 임베딩 서버 미연결(EMBED_OK=False) — 클러스터링은 서버 연결 시 실행됩니다.")


**출력 해석:** 50개 FAQ 임베딩을 5개 군집으로 묶고 **그래프 2개**로 보여 줍니다. 두 패널은 같은 현상을 다른 방식으로 보여 줍니다. 임베딩은 세부 5분류보다 **더 큰 의미 경계(HR ↔ IT·보안)**를 먼저 포착했습니다.

- 왼쪽 **PCA 산점도**(384→2차원, 색=군집): **군집 0(파랑, 32개)은 오른쪽, 군집 1(초록, 15개)은 왼쪽**으로 뚜렷이 나뉩니다. 단일 이상치(군집 2·3·4 = brown·gray·cyan)는 왼쪽 IT·보안 영역의 가장자리에 흩어져 있습니다.
- 오른쪽 **군집×카테고리 히트맵**(셀=FAQ 수): 산점도의 좌우 분리가 *어떤 카테고리*에 해당하는지 보여 줍니다.
  - **군집 1(15)** = `it`(7) + `security`(8): "기술·보안" 묶음입니다(산점도 왼쪽 초록).
  - **군집 0(32)** = `benefits`·`leave`·`payroll` 각 10 (+ `it`·`security` 각 1): "HR 제도(복지·휴가·급여)" 묶음입니다(산점도 오른쪽 파랑).
  - 군집 2·3·4 = 단일 이상치 각 1개(`security`·`it`·`it`)입니다. 큰 군집에 붙지 않은 가장자리 항목으로 볼 수 있습니다.
- 교훈: 임베딩은 **의미 유사도**를 잘 잡지만, 사람이 붙인 카테고리가 임베딩 공간보다 **더 세밀하게** 나뉘는 경우가 있습니다. `benefits`/`leave`/`payroll`은 모두 'HR 제도'에 가까워 한 군집으로 합쳐졌습니다. `cluster sizes` 쏠림({0:32,1:15,…})과 `linkage="average"`가 큰 군집을 만들기 쉬운 경향도 같은 맥락입니다.

> `scikit-learn`·`matplotlib`·임베딩 서버 중 하나라도 없으면 이 단계는 `[SKIP]`됩니다. 모두 선택 단계이므로 정상입니다.


### Session 1-5. 산출물 — `cluster_report.json`

**하는 일:** 앞 단계 결과를 `cluster_report.json`에 저장합니다.

**정상:** 저장 경로가 출력됩니다.

**의미:** 이 파일은 다음 Session이나 회귀 테스트의 입력으로 사용할 수 있습니다.


In [ ]:
from datetime import datetime, timezone

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "embedding_server_ok": EMBED_OK,
    "model": embedder.model if embedder else None,
    "dimension": DIM if EMBED_OK else None,
    "faq_count": len(records),
    "duplicate_pair_count": len(duplicates),
    "duplicate_threshold": DUPLICATE_THRESHOLD,
    "cluster_count": len(set(cluster_labels)) if cluster_labels else 0,
    "cluster_sizes": (
        {str(k): cluster_labels.count(k) for k in sorted(set(cluster_labels))}
        if cluster_labels
        else {}
    ),
    "sample_query": "연차는 몇 일인가요?",
}
path = out_dir / "cluster_report.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())

**출력 해석:** 저장 경로가 출력되면 `cluster_report.json`이 생성된 것입니다. 임베딩 서버가 없었던 경우에는 검색·클러스터링 결과가 비어 있을 수 있지만, 파일 생성 흐름 자체는 확인할 수 있습니다.


## Session 2. in-memory RAG — 인용이 핵심

**테스트 시나리오** — `internal_manual_snippets.json`의 조직 정책 스니펫(`manual`)과 고정 QA 5개 질문을 사용합니다. `retrieve`로 근거를 찾고, JSON 답변에 인용 `sources`를 함께 담는 흐름을 확인합니다.

Postgres 없이도 in-memory RAG의 동작을 그대로 관찰할 수 있습니다.


### Session 2-1. 매뉴얼 스니펫 → in-memory `VectorRetrievalStrategy`

**하는 일:** 조직 매뉴얼 스니펫 8건을 임베딩해 in-memory 벡터 인덱스를 만들고, `embed_fn`과 `search_fn`으로 `VectorRetrievalStrategy`를 구성한 뒤 샘플 질의로 동작을 확인합니다.

**의미:** `embed_fn`과 `search_fn`만 제공하면 `VectorRetrievalStrategy`가 표준 검색 인터페이스를 제공합니다. 여기서는 Session 1의 코사인 랭킹(`rank_by_query_vector`)을 `search_fn`으로 재사용합니다.


In [ ]:
manual = json.loads(
    (DATA / "internal_manual_snippets.json").read_text(encoding="utf-8")
)
snippets = manual["snippets"]
print(
    "시나리오: 조직 매뉴얼 스니펫", len(snippets), "건 (internal_manual_snippets.json)"
)
for s in snippets[:2]:
    print(
        f"  {s.get('id', '?')}:", (s.get("text") or s.get("content") or "")[:50] + "…"
    )


# (en) Build a tiny in-memory vector index from embedded snippets.
# (kr) 임베딩한 스니펫으로 작은 in-memory 벡터 인덱스를 만든다.
def build_memory_vector_strategy(emb, items, top_k_default=5):
    item_texts = [it["text"] for it in items]
    vectors = emb.embed_batch(item_texts)
    item_ids = [it["id"] for it in items]
    sources = [it["source"] for it in items]

    def embed_fn(text):
        return emb.embed_one(text)

    def search_fn(query_vec, top_k):
        k = top_k or top_k_default
        out = []
        for doc_id, score in rank_by_query_vector(
            query_vec, vectors, item_ids, top_k=k
        ):
            idx = item_ids.index(doc_id)
            out.append(
                {
                    "text": item_texts[idx],
                    "score": score,
                    "metadata": {"source": sources[idx], "doc_id": doc_id},
                }
            )
        return out

    return exaone.retrieval.VectorRetrievalStrategy(
        embed_fn=embed_fn, search_fn=search_fn
    )


retrieval = None
if EMBED_OK:
    retrieval = build_memory_vector_strategy(embedder, snippets)
    for h in retrieval.retrieve("재택근무 VPN 접속 방법", top_k=2):
        print(f"[{h.score:.3f}] {h.metadata.get('source')}: {h.text[:50]}")
else:
    # (en) Without an embedder we cannot build the in-memory index.
    # (kr) 임베더가 없으면 in-memory 인덱스를 만들 수 없다.
    print(
        "[SKIP] 임베딩 서버 미연결(EMBED_OK=False) — in-memory 검색 전략은 서버 연결 시 구성됩니다."
    )

**출력 해석:** Session 1의 FAQ 대신 이제 **조직 매뉴얼 스니펫 8건**을 지식원으로 사용합니다. 각 항목은 정책 문서 조각이며 `source`를 함께 가지고 있습니다.

- 테스트 질의 "재택근무 VPN 접속 방법"의 상위 2건이 **모두 `it-vpn-guide` 출처**(0.900·0.860)입니다. VPN 접속 절차와 사전 보안 요건 청크가 정확히 검색되었습니다.
- 핵심은 각 결과가 `score`·`text`뿐 아니라 **`source`(출처)**를 함께 가져온다는 점입니다. 이 출처가 Session 2-3의 **인용(`sources`)** 근거가 됩니다. 즉, RAG가 "근거 있는 답"을 만들기 위한 토대입니다.
- 이 `VectorRetrievalStrategy`는 Session 1의 코사인 `rank_by_query_vector`를 `search_fn`으로 **재사용**합니다. 같은 유사도 검색을 에이전트가 사용할 수 있는 검색 전략 객체로 감싼 것입니다.

> 임베딩 서버가 없으면(`EMBED_OK=False`) 이 셀은 `[SKIP]` 안내만 출력합니다. 오프라인으로 개념만 확인하는 단계에서는 정상 동작입니다.


### Session 2-2. `retrieve` 도구만 단독 확인 (LLM 없이)

**하는 일:** 5개 질문에 대해 `retrieve` 도구만 단독 호출하고(LLM 없이), 질문별 `chunk_count`와 `preview`를 출력합니다.

**의미:** `build_rag_tool_registry`는 검색 전략을 `retrieve` 도구로 감쌉니다. LLM을 부르기 전에 도구가 어떤 청크를 모으는지 먼저 확인해야 검색 품질 문제와 생성 품질 문제를 분리해서 볼 수 있습니다.


In [ ]:
rag_registry = exaone.agents.build_rag_tool_registry(retrieval) if retrieval else None
inspect_rows = []
if rag_registry is not None:
    for q in manual["questions"]:
        out = rag_registry.execute("retrieve", {"query": q, "top_k": 3})
        inspect_rows.append({"question": q, "tool_result": out})
        print("Q:", q)
        print("  chunk_count:", out.get("chunk_count"))
        print("  preview:", (out.get("content") or "")[:120].replace("\n", " "))
else:
    # (en) The retrieve tool needs the vector strategy from Session 2-1.
    # (kr) retrieve 도구는 Session 2-1의 벡터 전략이 있어야 한다.
    print(
        "[SKIP] 검색 전략 미구성(EMBED_OK=False) — retrieve 도구 점검은 임베딩 서버 연결 시 실행됩니다."
    )

**출력 해석:** LLM을 부르기 전에 `retrieve` 도구만 단독 호출해, 도구가 모으는 컨텍스트를 먼저 확인합니다.

- 5개 질문 모두 `chunk_count: 3`(`top_k=3`)이고, **각 질문의 #1 청크가 정확히 맞는 출처**입니다. 연차→`hr-leave-policy`, VPN→`it-vpn-guide`, 피싱→`security-phishing`, 급여→`payroll-calendar`, 자기계발비→`benefits-education`처럼 질문과 근거가 1:1로 잘 연결되었습니다.
- `preview`의 `<chunk source="…">`가 곧 LLM에 들어갈 컨텍스트이며, 이 `source`가 2-3 **인용**의 근거가 됩니다.
- 검색이 이렇게 정확하면 2-3에서 LLM은 올바른 근거를 바탕으로 답할 수 있습니다. **검색 품질이 답변 품질의 상한**이므로, 먼저 검색 도구의 결과를 확인합니다.

> 임베딩 서버가 없으면(`EMBED_OK=False`) 검색 전략이 없으므로 이 셀은 `[SKIP]`만 출력합니다. 정상 동작입니다.


### Session 2-3. LLM QA 5건 — 인용 포함 답변

**하는 일:** 질문 5개마다 `retrieve`로 청크를 가져와 `<retrieved_context>`에 넣고, `DEFAULT_SYSTEM_PROMPT_RAG`로 LLM이 그 근거만 사용해 JSON 답변(`answer`·`sources`)을 생성하게 합니다(`EXAONE_API_KEY` 필요).

**의미:** 검색을 코드에서 직접 실행해 모든 질문이 근거 청크를 바탕으로 답하도록 만듭니다. 이렇게 해야 RAG의 핵심인 *인용 있는 답변*을 안정적으로 시연할 수 있습니다. 자율 `ToolAgent(retrieval_strategy=…)`는 검색을 *도구*로 노출하고, 라우터와 모델이 도구 호출 여부를 판단합니다. 따라서 **검색 기능이 있어도 답변이 그 결과를 실제로 사용한다는 보장은 없으므로**, 인용을 보장해야 하는 이 데모에서는 검색을 직접 실행합니다.


In [ ]:
# (en) Reliable citation demo: run retrieval in code, SHOW the chunks, then inject the
#      SAME chunks as <retrieved_context> so the LLM grounds on them. The autonomous
#      ToolAgent exposes retrieve as a *tool* and (with the router) may skip it, so
#      citations are unreliable; driving retrieval directly makes the chain visible.
# (kr) 신뢰할 수 있는 인용 데모: 검색을 코드에서 직접 실행해 청크를 화면에 보여 주고, 같은
#      청크를 <retrieved_context>에 넣어 LLM이 그 근거로 답하게 한다. 자율 ToolAgent는
#      retrieve를 *도구*로 노출하고 라우터 판단에 따라 건너뛸 수 있어 인용이 안정적이지 않다.
import exaone.agents.rag_context

llm = None
qa_results = []
if retrieval is not None and HAS_API:
    try:
        llm = exaone.integrations.llm_env.build_llm_from_env()
    except SystemExit:
        print("LLM 환경이 불완전합니다 — EXAONE_* 값을 확인하세요.")

if retrieval is not None and llm is not None:
    # (en) Validate the JSON answer shape: require answer + sources (RAG grounding fields).
    # (kr) JSON 답변 형태를 검증한다: answer + sources(RAG 근거 필드)를 필수로 둔다.
    pipeline = exaone.output.StructuredOutputPipeline(required_keys=["answer", "sources"])
    # (en) Fill the {{CURRENT_DATE_UTC}} placeholder the way the agent does at run time.
    # (kr) 에이전트가 런타임에 채우는 {{CURRENT_DATE_UTC}} placeholder를 동일하게 채운다.
    utc_now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    rag_system = exaone.agents.prompts.DEFAULT_SYSTEM_PROMPT_RAG.replace(
        exaone.agents.prompts.CURRENT_DATE_UTC_PLACEHOLDER, utc_now
    )
    for q in manual["questions"]:
        # (en) Retrieve once: show these chunks AND format the same ones into the prompt.
        # (kr) 한 번 검색해: 이 청크들을 화면에 보여 주고, 같은 청크를 프롬프트에 넣는다.
        hits = retrieval.retrieve(q, top_k=3)
        context_block = exaone.agents.rag_context.format_retrieved_chunks(hits, max_chars=4000)
        user_msg = exaone.agents.prompts.format_rag_user_message(context_block, q)
        messages = [
            exaone.llm.ExaoneMessage(role="system", content=rag_system),
            exaone.llm.ExaoneMessage(role="user", content=user_msg),
        ]
        resp = llm.chat(messages, options=exaone.llm.ExaoneGenerateOptions(enable_thinking=False))
        structured = pipeline.process(resp.content or "")
        data = structured.data if isinstance(structured.data, dict) else {}
        qa_results.append({
            "question": q,
            "answer": data.get("answer") or resp.content,
            "sources": data.get("sources") or [],
            "chunk_count": len(hits),
            "error": structured.error,
        })
        print("Q:", q)
        print("  검색된 청크 (top 3 → 이게 근거로 LLM에 들어감):")
        for h in hits:
            print(f"    [{h.score:.3f}] {h.metadata.get('source')}: {h.text[:55]}")
        print("  answer:", (qa_results[-1]["answer"] or "")[:100])
        print("  sources:", qa_results[-1]["sources"])
        # (en) Grounding signal: chunks retrieved but no citation -> model may not have grounded.
        # (kr) 근거 신호: 청크는 검색됐는데 인용이 비면 모델이 근거를 사용하지 못했을 수 있다.
        if not qa_results[-1]["sources"] and len(hits) > 0:
            print("  ⚠ 근거 인용 비어 있음 — 답이 검색 근거로 뒷받침되지 않았을 수 있음")
        print()
else:
    reason = "검색 전략 미구성(임베딩 서버)" if retrieval is None else "EXAONE_API_KEY 미설정"
    print(f"[SKIP] {reason} — 인용 QA는 임베딩 서버 + API 키가 모두 있을 때 실행됩니다.")


**출력 해석:** 이제 각 질문에 대해 `retrieve`로 찾은 청크를 `<retrieved_context>`에 넣어 LLM에 전달합니다. 따라서 답변을 만들 때 항상 검색 근거를 함께 참고하게 됩니다. 이번 라이브 실행에서는 **5개 질문 모두 출처 인용에 성공**했습니다.

- 각 질문 아래에 **검색된 청크(top 3 — 점수·출처·미리보기)**가 답변 위에 함께 출력되어, `질문 → 검색된 근거 → 답변 → 인용`의 전체 흐름을 한 셀에서 볼 수 있습니다.
- 검색은 완벽하지 않습니다 — 예: '연차' 질문의 top-3 에 `payroll`·반차 청크도 섞여 들어옵니다(점수는 더 낮음). 그래도 모델은 **가장 관련 있는 #1 청크**를 근거로 답하고 그것만 인용합니다 — 약간의 noise 가 섞여도 grounding 이 핵심 근거를 고르는 모습입니다.
- 5개 답변 모두 **근거 청크를 바탕으로 한국어 높임말로 작성**되었고, `sources` 필드도 채워졌습니다. 예를 들어 "급여는 매월 25일에 지급되며…"는 `payroll-calendar`, "자기계발비 한도는 연 200만 원입니다"는 `benefits-education`을 출처로 인용합니다. 2-2에서 확인한 출처가 답변의 인용으로 그대로 이어집니다.
- `sources`가 있다는 것은 모델이 검색 결과를 참고했다는 **좋은 신호**입니다. 다만 답변 내용이 실제 근거와 정확히 맞는다는 뜻은 아니므로, 위 코드는 근거 청크가 있는데도 `sources`가 비어 있으면 ⚠ 표시로 따로 알려줍니다.
- 한국어 컨텍스트를 명시적으로 넣었기 때문에 답변도 **한국어로** 안정적으로 생성됩니다. 앞선 자율 실행처럼 컨텍스트 없이 영어로 거절하던 경우와 대비됩니다.

> 임베딩 서버 또는 `EXAONE_API_KEY`가 없으면 이 셀은 `[SKIP]`만 출력합니다. 정상 동작입니다.


### Session 2-4. 산출물 — `manual_qa.json`

**하는 일:** 앞 단계 결과를 `manual_qa.json`에 저장합니다.

**정상:** 저장 경로가 출력됩니다.

**의미:** 이 파일은 다음 Session이나 회귀 테스트의 입력으로 사용할 수 있습니다.


In [ ]:
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "embedding_ok": EMBED_OK,
    "has_api": HAS_API,
    "inspect_rows": inspect_rows,
    "qa_results": qa_results,
}
path = out_dir / "manual_qa.json"
path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())

**출력 해석:** 저장 경로가 출력되면 `manual_qa.json`이 생성된 것입니다. 임베딩 서버나 API 키가 없어 QA가 건너뛰어진 경우에도, 파일에는 실행 가능한 범위의 결과가 기록됩니다.


## Session 3. RAG 실패 모드 — 라이브러리 API로 복구

**테스트 시나리오** — `failure_case_fixtures.json`의 실패 케이스 5건(`failure_cases`)을 사용합니다. 빈 검색, 미설정, 중복, 컨텍스트 초과, 프롬프트 주입을 **케이스 ID**로 고정해 재현합니다.

| id | Session | 검증 포인트 |
|---|---|---|
| `empty_retrieval` | 3-2 | 검색 결과 0건이 예외가 아님 |
| `not_configured` | 3-3 | 명시적 에러 |
| `duplicate_chunks` | 3-4 | `source` 중복 제거 |
| `context_overflow` | 3-5 | 토큰 상한 적용 |
| `injection_in_chunk` | 3-6 | 주입 시도 무력화 |

운영 환경에서 RAG가 흔히 실패하는 지점은 어느 정도 정해져 있습니다. 이 세션에서는 그 실패를 작은 케이스로 고정해 재현하고 복구합니다.


### Session 3-1. failure_cases 로드

**하는 일:** `failure_cases` 5건을 읽고 케이스 ID를 출력합니다.

**입력:** `data/failure_case_fixtures.json`

**정상:** `시나리오:` + `cases:` 5 + ID 5줄

**의미:** Session 3-2~3-6이 각 ID를 순서대로 검증합니다.


In [ ]:
import exaone.retrieval
import exaone.retrieval.errors
import exaone.retrieval.base_strategy
import exaone.agents.rag_context
import exaone.context_management

cases = json.loads((DATA / "failure_case_fixtures.json").read_text(encoding="utf-8"))
print("시나리오: RAG 실패 모드", len(cases), "건 (failure_case_fixtures.json)")
for c in cases:
    print(f"  {c['id']}")
print("cases:", [c["id"] for c in cases])

**출력 해석:** 운영 환경에서 RAG가 흔히 실패하는 **5가지 지점**을 케이스 ID로 고정해 불러옵니다.

- `empty_retrieval`(검색 결과 0건)·`not_configured`(미설정)·`duplicate_chunks`(중복)·`context_overflow`(컨텍스트 초과)·`injection_in_chunk`(주입 시도)를 각각 3-2~3-6에서 검증합니다.
- 케이스 ID로 고정했으므로 LLM 호출 없이도 같은 상황을 **항상 같은 방식으로** 재현하고 회귀 테스트할 수 있습니다.


### Session 3-2. Case 1 — 빈 검색은 예외가 아니라 빈 결과로

**하는 일:** 검색 결과가 없을 때도 예외를 던지지 않고 빈 결과 페이로드를 반환하는지 확인합니다.

**정상:** `case1 pass: True`가 출력됩니다.

**의미:** 검색 결과 없음은 장애가 아니라 정상적으로 처리해야 할 상태입니다. 그래야 뒤의 답변 단계에서 "근거 없음"을 명확히 다룰 수 있습니다.


In [ ]:
empty_strategy = exaone.retrieval.VectorRetrievalStrategy(
    embed_fn=lambda _q: [0.0] * 8, search_fn=lambda _v, _k: []
)
reg = exaone.agents.build_rag_tool_registry(empty_strategy)
out_empty = reg.execute("retrieve", {"query": cases[0]["query"]})
case1_pass = out_empty.get("chunk_count") == 0 and "[No retrieval hits]" in (
    out_empty.get("content") or ""
)
print("case1 pass:", case1_pass, "|", out_empty)

**출력 해석:** 검색이 **0건**일 때 `retrieve`가 예외를 던지지 않고 정상 페이로드를 반환합니다.

- `chunk_count: 0` + `content: "[No retrieval hits]"`는 빈 결과를 *에러가 아니라 데이터*로 다룬다는 뜻입니다. 파이프라인이 멈추지 않고, 뒤의 LLM 단계는 "근거 없음"으로 정직하게 답할 수 있습니다.
- `case1 pass: True`는 이 두 조건(0건 + 명시 표식)을 확인합니다. 운영에서 가장 흔한 "검색 결과 없음" 상황을 실행 중단 없이 안전하게 처리함을 보여줍니다.


### Session 3-3. Case 2 — 미설정 전략은 명시적 에러로

**하는 일:** 검색 전략에 필요한 함수가 연결되지 않았을 때 명시적인 설정 오류가 발생하는지 확인합니다.

**정상:** `case2 pass: True`가 출력됩니다.

**의미:** 검색 전략이 잘못 연결된 상태를 빈 결과처럼 넘기면 원인 파악이 어려워집니다. 설정 오류는 초기에 분명하게 드러나야 합니다.


In [ ]:
case2_pass = False
raised = None
# (en) Capture the error message too — the point of this case is the *explicit*, actionable error.
# (kr) 에러 메시지도 함께 확인한다. 이 케이스의 핵심은 *명시적이고 조치 가능한* 에러다.
try:
    exaone.retrieval.VectorRetrievalStrategy()
except exaone.retrieval.errors.RetrievalNotConfiguredError as exc:
    case2_pass = True
    raised = f"{type(exc).__name__}: {exc}"
print("case2 pass:", case2_pass)
print("raised:", raised)


**출력 해석:** `embed_fn`·`search_fn` 없이 `VectorRetrievalStrategy()`를 만들면 **명시적 에러**(`RetrievalNotConfiguredError`)가 발생합니다.

- `raised:`에 찍힌 메시지는 단순히 "에러"라고만 말하지 않고 **무엇을 어떻게 고쳐야 하는지**까지 알려줍니다("requires embed_fn and search_fn. Wire PgVectorAdapter…"). 미설정 전략이 나중에 모호한 `AttributeError`나 빈 결과로 새지 않고 **설정 시점에 즉시 실패(fail-fast)**합니다.
- `case2 pass: True`는 그 예외가 실제로 발생함을 확인합니다. 잘못된 연결 설정을 조용히 넘기지 않고 즉시 드러내는 것이 운영상 안전합니다.


### Session 3-4. Case 3 — 같은 `source`의 중복 청크 제거

**하는 일:** 같은 `source`에서 온 중복 청크를 하나로 정리하는지 확인합니다.

**정상:** `case3 pass: True`가 출력됩니다.

**의미:** 같은 문서 조각이 반복되면 토큰을 낭비하고 답변이 한 근거에 과도하게 치우칠 수 있습니다. 인용 전에 중복을 줄이는 것이 좋습니다.


In [ ]:
RetrievalResult = exaone.retrieval.base_strategy.RetrievalResult
dup_chunks = [
RetrievalResult(text="VPN은 2FA 후 접속", score=0.9, metadata={"source": "it-vpn"}),
RetrievalResult(text="VPN은 2FA 후 접속(중복)", score=0.88, metadata={"source": "it-vpn"}),
RetrievalResult(text="연차는 HR 포털", score=0.7, metadata={"source": "hr-leave"}),
]
seen = set()
deduped = []
for ch in dup_chunks:
    src = (ch.metadata or {}).get("source") or ch.text[:20]
    if src in seen:
        continue
    seen.add(src)
    deduped.append(ch)
formatted = exaone.agents.rag_context.format_retrieved_chunks(deduped, max_chars=4000)
case3_pass = formatted.count('source="it-vpn"') == 1
print("case3 pass:", case3_pass)
# (en) Show the dedup tangibly: same-source chunks collapse instead of padding the context.
# (kr) 중복 제거를 눈에 보이게 보여준다. source가 같은 청크를 합쳐 컨텍스트를 늘리지 않는다.
before_sources = [(c.metadata or {}).get("source") for c in dup_chunks]
after_sources = [(c.metadata or {}).get("source") for c in deduped]
print(f"  chunks: {len(dup_chunks)} → {len(deduped)} (출처 중복 제거)")
print(f"  sources: {before_sources} → {after_sources}")


**출력 해석:** 같은 `source`의 중복 청크를 **하나로 합칩니다**. 출력의 `chunks: 3 → 2`, `sources: [it-vpn, it-vpn, hr-leave] → [it-vpn, hr-leave]`가 그 과정을 보여줍니다.

- `case3 pass: True`는 정리된 컨텍스트에 `source="it-vpn"`가 정확히 1번만 남았는지 확인합니다. 중복 청크는 토큰을 낭비하고 답을 한쪽으로 치우치게 하므로 인용 전에 제거합니다.
- 검색은 같은 문서의 인접 조각을 여러 번 가져오기 쉽습니다. 그래서 `source` 기준 중복 제거는 RAG 위생의 기본입니다.


### Session 3-5. Case 4 — 과도한 컨텍스트는 토큰 상한으로 제한

**하는 일:** 컨텍스트가 토큰 예산을 넘을 때 상한 이하로 줄이는지 확인합니다.

**정상:** `case4 pass: True`가 출력됩니다.

**의미:** 검색 결과가 너무 길면 모델 호출 자체가 실패할 수 있습니다. 컨텍스트를 토큰 예산 안으로 줄이는 안전장치가 필요합니다.


In [ ]:
long_text = cases[3]["long_context"]
max_input = 512
capped = exaone.context_management.hard_cap_messages(
    [{"role": "user", "content": long_text}], max_input_tokens=max_input
)
after_tokens = exaone.context_management.estimate_tokens_from_messages(capped)
case4_pass = after_tokens <= max_input
print("case4 pass:", case4_pass, "| tokens:", after_tokens, "(<=", max_input, ")")

**출력 해석:** 토큰 예산을 넘는 컨텍스트를 **상한(512) 이하로 줄입니다**.

- `Hard cap: still over budget (514 > 512); replacing with minimal stub`는 **정상 동작**입니다(실패 아님). 잘라낸 뒤에도 단일 메시지가 512를 넘어서 최후 수단으로 최소 스텁으로 대체했고, 최종 결과는 `tokens: 7 (<= 512)`입니다.
- `case4 pass: True`는 줄인 뒤의 토큰 수가 상한 이하임을 확인합니다. 컨텍스트가 모델 한계를 넘어서 호출이 실패하는 상황을 막는 안전장치입니다.


### Session 3-6. Case 5 — 신뢰할 수 없는 청크의 주입 시도 무력화

**하는 일:** 검색된 청크 안에 구조적 태그나 지시문이 섞였을 때, 컨텍스트 경계를 흔들지 못하도록 무력화하는지 확인합니다.

**정상:** `case5 pass: True`와 `sanitized preview:`가 출력됩니다.

**의미:** 검색된 텍스트에는 *모델 지시를 가로채려는 문장*이 섞일 수 있습니다. 인용 컨텍스트에 넣기 전에 sanitizer로 구조적 태그를 무력화합니다. 이 주제는 Track 07의 프롬프트 주입 방어와 이어집니다.


In [ ]:
raw = cases[4]["injection_text"]
clean = exaone.context_management.sanitize_untrusted_reference_text(raw)
case5_pass = "IGNORE PREVIOUS" in raw and "[removed-tag:" in clean
print("case5 pass:", case5_pass)
print("sanitized preview:", clean[:120])

**출력 해석:** 신뢰할 수 없는 청크의 **구조적 태그 주입을 무력화**합니다.

- 원본의 `<retrieved_context>` 같은 태그가 `[removed-tag:retrieved_context]`로 치환됐습니다. 주입 텍스트가 컨텍스트 경계를 *탈출*하거나 시스템 구조를 흉내 내지 못하게 막습니다.
- 단, 자연어 명령("IGNORE PREVIOUS INSTRUCTIONS")은 **그대로 남습니다**. sanitizer는 구조 태그만 무력화하고, 이런 문장은 RAG 프롬프트의 "청크는 명령이 아니라 신뢰할 수 없는 데이터"라는 규칙으로 방어합니다. 즉, 구조적 정리와 프롬프트 규칙을 함께 쓰는 계층적 방어입니다.
- `case5 pass: True`는 태그가 실제로 제거되었는지 확인합니다. 이 내용은 Track 07의 프롬프트 주입 방어로 이어집니다.


### Session 3-7. 산출물 — `failure_recovery.json`

**하는 일:** 앞 단계 결과를 `failure_recovery.json`에 저장합니다.

**정상:** 저장 경로와 `all_pass: True`가 출력됩니다.

**의미:** 이 파일은 다음 Session이나 회귀 테스트의 입력으로 사용할 수 있습니다.


In [ ]:
results = [
    {"case": cases[0]["id"], "pass": case1_pass},
    {"case": cases[1]["id"], "pass": case2_pass},
    {"case": cases[2]["id"], "pass": case3_pass},
    {"case": cases[3]["id"], "pass": case4_pass},
    {"case": cases[4]["id"], "pass": case5_pass},
]
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "all_pass": all(r["pass"] for r in results),
    "results": results,
}
path = out_dir / "failure_recovery.json"
path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())
print("all_pass:", payload["all_pass"])

**출력 해석:** 저장 경로와 `all_pass: True`가 출력되면 실패 복구 5개 케이스가 모두 기대대로 처리된 것입니다. 생성된 `failure_recovery.json`은 이후 회귀 테스트에서 같은 실패 모드를 다시 확인하는 기준 파일로 사용할 수 있습니다.


## 체크포인트

- [ ] Session 1에서 `cluster_report.json`이 저장됩니다. 임베딩 서버가 없으면 차원·중복 결과가 비어 있어도 정상입니다.
- [ ] Session 2에서 `retrieve` 점검 결과가 질문별 청크를 모읍니다. API 키가 있으면 QA 답변의 `sources`도 채워집니다.
- [ ] Session 3의 실패 복구 5종이 **모두 통과**합니다(`all_pass: true`). 이 단계는 임베딩 서버와 API 키 없이도 실행됩니다.

**다음:** Track 05 — Memory & Long Context (또는 심화 `04b_pgvector_and_strategies`)
